In [1]:
import numpy as np
import pandas as pd
import torch
import os

import sys
from pathlib import Path
module_path = Path.cwd().parent / 'utils'
sys.path.append(str(module_path))
import metrics
from simulations import *
module_path = Path.cwd().parent / 'execute'
sys.path.append(str(module_path))

import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

/opt/anaconda3/envs/env_synthcity/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[KeOps] Warning : omp.h header is not in the path, disabling OpenMP. To fix this, you can set the environment
                  variable OMP_PATH to the location of the header before importing keopscore or pykeops,
                  e.g. using os.environ: import os; os.environ['OMP_PATH'] = '/path/to/omp/header'
[KeOps] Warning : Cuda libraries were not detected on the system or could not be loaded ; using cpu only mode


In [2]:
def adjust_feat_types_for_generator(generator_name, feat_types_dict):
    feat_types_dict_ext = [dict(ft) for ft in feat_types_dict]  # deep copy
    for d in feat_types_dict_ext:
        if d['name'] == "survcens":
            if "HI-VAE_weibull" in generator_name: # == "HI-VAE_weibull" or generator_name == "HI-VAE_weibull_prior":
                d["type"] = 'surv_weibull'
            elif generator_name == "HI-VAE_lognormal":
                d["type"] = 'surv'
            else:
                d["type"] = 'surv_piecewise'
    return feat_types_dict_ext

In [3]:
# ---------- Load saved original-control and PROCOVA results ----------
n_samples = 3000
aug_perc = 1 / 3
alpha = 0.05

current_path = Path.cwd()
project_path = current_path.parent if current_path.name == "tutorials" else current_path
results_dir = (project_path/"dataset"/"Simulations_aug_indep_traincontrol"/f"results_two_stage_N{n_samples}")
result_files = sorted(results_dir.glob("results_MC_*.csv"))

results_original_control = pd.concat((pd.read_csv(path, index_col=0) for path in result_files), ignore_index=True,)
results_original_control["aug_perc"] = pd.to_numeric(results_original_control["aug_perc"], errors="coerce")
results_original_control = results_original_control[np.isclose(results_original_control["aug_perc"], aug_perc)].copy()

baseline_metrics = {
    "Log-rank": "log_pvalue_init",
    "Adjusted_Log-rank": "log_pvalue_adjusted_logrank_init",
    "Prog_Adjusted_Log-rank": "log_pvalue_prog_adjusted_logrank_init",
    "Cox_Wald_X": "log_pvalue_cox_wald_x_init",
    "Cox_Wald_Prog": "log_pvalue_cox_wald_prog_init",
}
required_columns = ["MC_seed", "H0_coef", *baseline_metrics.values()]
missing_columns = set(required_columns) - set(results_original_control.columns)
if missing_columns:
    raise ValueError(f"Saved results are missing columns: {sorted(missing_columns)}")

# Baseline metrics repeat for every synthetic draw; keep one row per MC/effect.
baseline_results = results_original_control.drop_duplicates(subset=["MC_seed", "H0_coef", "aug_perc"]).copy()
baseline_long = baseline_results[["H0_coef", *baseline_metrics.values()]].melt(id_vars="H0_coef", var_name="metric_column", value_name="log_pvalue",)
baseline_long["method"] = baseline_long["metric_column"].map({column: method for method, column in baseline_metrics.items()})
baseline_values = pd.to_numeric(baseline_long["log_pvalue"], errors="coerce")
baseline_long["reject"] = (baseline_values > -np.log(alpha)).where(baseline_values.notna())

summary = (baseline_long.groupby(["H0_coef", "method"]).agg(rejection_rate=("reject", "mean"), n_tests=("reject", "count"),)
    .reset_index()
    .rename(columns={"H0_coef": "treatment_effect"})
)

type1_error = summary[np.isclose(summary["treatment_effect"], 0.0)]
print("Empirical Type-I error from saved results:")
print(type1_error[["method", "rejection_rate", "n_tests"]])

Empirical Type-I error from saved results:
                   method  rejection_rate  n_tests
0       Adjusted_Log-rank        0.055556       90
1           Cox_Wald_Prog        0.055556       90
2              Cox_Wald_X        0.022222       90
3                Log-rank        0.022222       90
4  Prog_Adjusted_Log-rank        0.055556       90


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font="STIXGeneral", context="talk")

# ---------- Configuration ----------
aug_perc = 1 / 3
alpha = 0.05

# One outcome test for every generative model; baseline curves show all tests.
GENERATOR_TEST = "Log-rank"
GENERATOR_TEST_PREFIXES = {
    "Log-rank": "log_pvalue",
    "Adjusted_Log-rank": "log_pvalue_adjusted_logrank_D",
    "Prog_Adjusted_Log-rank": "log_pvalue_prog_adjusted_logrank_D",
    "Cox_Wald_X": "log_pvalue_cox_wald_x_D",
    "Cox_Wald_Prog": "log_pvalue_cox_wald_prog_D",
}
def generator_test_column(generator):
    return f"{GENERATOR_TEST_PREFIXES[GENERATOR_TEST]}_{generator}"

generators_sel = [
    # "HI-VAE_weibull",
    # "HI-VAE_piecewise",
    "HI-VAE_weibull_prior",
    "HI-VAE_piecewise_prior",
    "Surv-GAN",
    "Surv-VAE",
    "TabPFN-naive-full",
    "TabPFN-naive-control",
    "TabFORGE-naive",
]

current_path = Path.cwd()
project_path = current_path.parent if current_path.name == "tutorials" else current_path
folder_path_traincontrol = project_path / "dataset" / "Simulations_aug_indep_traincontrol"
folder_path_trainfull = project_path / "dataset" / "Simulations_aug_indep_trainfull"
base_results_dir = folder_path_traincontrol / f"results_two_stage_N{n_samples}_bis"
# Explicit paths keep the two training experiments separate.
tabpfn_full_results_dir = folder_path_trainfull / f"results_two_stage_TabPFN_N{n_samples}_control1of3_bis"
tabpfn_control_results_dir = folder_path_traincontrol / f"results_two_stage_TabPFN_N{n_samples}_control1of3"

base_result_files = sorted(base_results_dir.glob("results_MC_*.csv"))
if not base_result_files:
    raise FileNotFoundError(f"No simulation results found in {base_results_dir}")
results_control_indep = pd.concat(
    (pd.read_csv(path, index_col=0) for path in base_result_files), ignore_index=True)


def load_generator_results(directory, label, effect_subdirs=False, stored_name="TabPFN-naive"):
    """Load one experiment and give all its generator metric columns a unique suffix."""
    pattern = "effect_*/results_MC_*.csv" if effect_subdirs else "results_MC_*.csv"
    files = sorted(directory.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No {label} results found in {directory}/{pattern}")
    frame = pd.concat((pd.read_csv(path, index_col=0) for path in files), ignore_index=True)
    identity = ["MC_seed", "H0_coef", "aug_perc", "synthetic_id"]
    missing = set(identity + [f"log_pvalue_{stored_name}"]) - set(frame.columns)
    if missing:
        raise ValueError(f"{label} results are missing columns: {sorted(missing)}")
    for column in identity:
        frame[column] = pd.to_numeric(frame[column], errors="raise")
    if frame[identity].isna().any().any():
        raise ValueError(f"{label} results contain missing MC/effect/draw identifiers.")
    if frame.duplicated(identity).any():
        raise ValueError(f"Overlapping {label} shards contain duplicate MC/effect/draw rows.")
    # Rename outcome AND selection-score columns, leaving the CSVs unchanged.
    suffix = "_" + stored_name
    frame = frame.rename(columns={
        column: column[:-len(suffix)] + "_" + label
        for column in frame.columns if column.endswith(suffix)
    })
    frame = frame[np.isclose(frame["aug_perc"], aug_perc)].copy()
    if frame.empty:
        raise ValueError(f"No {label} results found for aug_perc={aug_perc}")
    return frame


separate_sources = {
    "TabPFN-naive-full": load_generator_results(
        tabpfn_full_results_dir, "TabPFN-naive-full", effect_subdirs=True),
    "TabPFN-naive-control": load_generator_results(
        tabpfn_control_results_dir, "TabPFN-naive-control"),
}
separate_sources["TabFORGE-naive"] = load_generator_results(
    folder_path_traincontrol / f"results_two_stage_TabFORGE_N{n_samples}_control1of3",
    "TabFORGE-naive", stored_name="TabFORGE-naive")
print("TabFORGE: preliminary scratch decoder/diffusion run; pretrained encoder. "
      "Synthetic draws within an MC seed are dependent; inspect MC counts below.")
for column in ["aug_perc", "H0_coef", "H0_coef_univ", "D_control", "D_treated"]:
    results_control_indep[column] = pd.to_numeric(results_control_indep[column], errors="coerce")
df_res = results_control_indep[np.isclose(results_control_indep["aug_perc"], aug_perc)].copy()
if df_res.empty:
    raise ValueError(f"No main simulation results found for aug_perc={aug_perc}")
for label, frame in separate_sources.items():
    missing_effects = [effect for effect in df_res["H0_coef"].unique()
                       if not np.isclose(frame["H0_coef"], effect).any()]
    if missing_effects:
        print(f"[missing] {label} treatment effects: {sorted(missing_effects)}")
    print(f"{label}: MC counts by treatment effect:")
    print(frame.groupby("H0_coef")["MC_seed"].nunique().to_string())


def generator_results(generator):
    """Route each displayed learner to its own experiment; never pool MC rows."""
    if generator in separate_sources:
        return separate_sources[generator]
    return df_res

# ---------- Empirical power of generators ----------
plot_data = []

for name in generators_sel:
    source_df = generator_results(name)
    col = generator_test_column(name)
    if col not in source_df.columns:
        print(f"[skip] {name}: missing {col}")
        continue
    source_df[col] = pd.to_numeric(source_df[col], errors="coerce")
    power = (source_df.assign(reject=(source_df[col] > -np.log(alpha)).where(source_df[col].notna())).groupby("H0_coef")["reject"].mean())
    for beta, value in power.items():
        plot_data.append({"Learner": name, "Treatment effect": beta, "Power": value})

# ---------- Theoretical power from the main simulation ----------
df_theo = df_res.groupby("H0_coef")[["H0_coef_univ", "D_control", "D_treated"]].mean()
for beta, row in df_theo.iterrows():
    theoretical_cases = {
        "Theoretical (original)": row["D_control"],
        "Theoretical (downsized)": row["D_control"] * aug_perc,
    }
    for name, n_control in theoretical_cases.items():
        power = cpower(n_control, row["D_treated"], row["H0_coef_univ"], alpha,)
        plot_data.append({"Learner": name, "Treatment effect": beta, "Power": power})

# ---------- Add saved original-control / PROCOVA tests ----------
# Baselines retain all five tests independently of GENERATOR_TEST.
statistical_methods = list(baseline_metrics)
df_adjusted = (summary[summary["method"].isin(statistical_methods)].rename(columns={
        "method": "Learner",
        "treatment_effect": "Treatment effect",
        "rejection_rate": "Power",
    })[["Learner", "Treatment effect", "Power"]].copy())

missing_methods = set(statistical_methods) - set(df_adjusted["Learner"])
if missing_methods:
    raise ValueError(f"Missing saved baseline methods: {sorted(missing_methods)}")

df_plot = pd.concat([pd.DataFrame(plot_data), df_adjusted], ignore_index=True,)
df_plot["Treatment effect"] = pd.to_numeric(df_plot["Treatment effect"], errors="coerce")
df_plot["Power"] = pd.to_numeric(df_plot["Power"], errors="coerce")
df_plot = df_plot.dropna(subset=["Treatment effect", "Power"])

# ---------- Plot styles ----------
styles = {
    "Theoretical (original)": {"color": "#0072B2", "ls": "-", "marker": "o"},
    "Theoretical (downsized)": {"color": "#009E73", "ls": "-", "marker": "o"},
    "HI-VAE_weibull": {"color": "#E69F00", "ls": "--", "marker": "s"},
    "HI-VAE_piecewise": {"color": "#D55E00", "ls": "--", "marker": "s"},
    "HI-VAE_weibull_prior": {"color": "#F0E442", "ls": "--", "marker": "s"},
    "HI-VAE_piecewise_prior": {"color": "#B21800", "ls": "--", "marker": "s"},
    "Surv-GAN": {"color": "#56E4E9", "ls": "--", "marker": "^"},
    "Surv-VAE": {"color": "#009E96", "ls": "--", "marker": "^"},
    "TabPFN-naive-full": {"color": "#041795", "ls": "--", "marker": "^"},
    "TabPFN-naive-control": {"color": "#7B61A8", "ls": "-.", "marker": "v"},
    "TabFORGE-naive": {"color": "#E66101", "ls": ":", "marker": "*"},
    "Log-rank": {"color": "#333333", "ls": ":", "marker": "D"},
    "Adjusted_Log-rank": {"color": "#17BECF", "ls": ":", "marker": "P"},
    "Prog_Adjusted_Log-rank": {"color": "#8C564B", "ls": ":", "marker": "X"},
    "Cox_Wald_X": {"color": "#CC79A7", "ls": "-.", "marker": "^"},
    "Cox_Wald_Prog": {"color": "#5E3C99", "ls": "-.", "marker": "v"},
}

# ---------- Plot all generated datasets ----------
fig, ax = plt.subplots(figsize=(9, 5.5))
for name, style in styles.items():
    data = df_plot[df_plot["Learner"] == name].sort_values("Treatment effect")
    if data.empty:
        continue
    ax.plot(
        data["Treatment effect"], data["Power"], label=name,
        color=style["color"], linestyle=style["ls"], marker=style["marker"],
        linewidth=2.2, markersize=7,
    )

ax.set_title(
    f"Independent simulation — Augmentation x3\nTest: {GENERATOR_TEST}",
    fontsize=16, fontweight="semibold",
)
ax.set_xlabel(r"Treatment effect ($\beta$)", fontsize=14)
ax.set_ylabel("Type I error / Power", fontsize=14)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0, 1.05)
ax.set_xticks(np.arange(0, 1.1, 0.2))
ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.tick_params(axis="both", labelsize=12)
ax.grid(True, alpha=0.3)
ax.axhline(alpha, color="gray", linestyle="--", linewidth=1, alpha=0.6, zorder=0)
ax.legend(
    loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3,
    fontsize=10, frameon=False, columnspacing=1.5, handlelength=3,
)
plt.show()

type1_error = (
    df_plot[np.isclose(df_plot["Treatment effect"], 0.0)]
    [["Learner", "Power"]]
    .rename(columns={"Power": "Type-I error"})
    .sort_values("Type-I error")
)
print("\nEmpirical Type-I error (treatment effect = 0):")
print(type1_error.to_string(index=False))

In [ ]:
# ---------- All selection methods: change only SELECTION_METHOD ----------
SELECTION_METHOD = "tableone"  # "logrank", "tableone", or "type1"
SELECTION_LABELS = {
    "logrank": "largest synthetic-vs-real-control log-rank p-value",
    "tableone": "largest TableOne minimum p-value",
    "type1": "synthetic-draw ID with null rejection rate closest to 0.05",
}
TYPE1_TARGET = 0.05
selection_keys = ["MC_seed", "H0_coef", "aug_perc"]


def select_generator_draws(source, generator, method):
    if method not in SELECTION_LABELS:
        raise ValueError(f"Unknown selection method: {method}")
    outcome = generator_test_column(generator)
    if outcome not in source:
        print(f"[skip] {generator}: missing {outcome}")
        return pd.DataFrame(), pd.DataFrame()
    data = source.copy()
    data[outcome] = pd.to_numeric(data[outcome], errors="coerce")
    diagnostics = pd.DataFrame()
    if method == "type1":
        keys = selection_keys + ["synthetic_id"]
        if data.duplicated(keys).any():
            raise ValueError(f"Duplicate MC/effect/draw rows for {generator}")
        null = data[np.isclose(data.H0_coef, 0)].copy()
        if null.empty:
            raise ValueError(f"No null MC replicates for {generator}")
        n_mc = null.MC_seed.nunique()
        null["reject"] = (null[outcome] > -np.log(alpha)).where(null[outcome].notna())
        diagnostics = null.groupby("synthetic_id").agg(
            type1_error=("reject", "mean"), n_valid=("reject", "count"))
        # Compare IDs on the same complete set of null replicates.
        diagnostics = diagnostics[diagnostics.n_valid == n_mc].reset_index()
        if diagnostics.empty:
            raise ValueError(f"No synthetic ID has complete null results for {generator}")
        diagnostics["distance_to_target"] = (diagnostics.type1_error - TYPE1_TARGET).abs()
        diagnostics = diagnostics.sort_values(["distance_to_target", "synthetic_id"])
        winning_id = diagnostics.iloc[0].synthetic_id
        diagnostics["selected"] = diagnostics.synthetic_id == winning_id
        diagnostics["Learner"] = generator
        selected = data[data.synthetic_id == winning_id].copy()
        if len(selected) != len(data[selection_keys].drop_duplicates()):
            raise ValueError(f"Selected ID {winning_id} is missing MC/effect rows for {generator}")
    else:
        score = (f"log_pvalue_control_{generator}" if method == "logrank"
                 else f"TableOne min p-value_{generator}")
        if score not in data:
            print(f"[skip] {generator}: missing {score}")
            return pd.DataFrame(), diagnostics
        data[score] = pd.to_numeric(data[score], errors="coerce")
        # Selection depends on its score, not on the outcome's availability.
        candidates = data.dropna(subset=[score])
        if candidates.empty:
            return pd.DataFrame(), diagnostics
        grouped = candidates.groupby(selection_keys, dropna=False)[score]
        indices = grouped.idxmin() if method == "logrank" else grouped.idxmax()
        selected = candidates.loc[indices].copy()
    selected["Learner"] = generator
    selected["selected_log_pvalue"] = selected[outcome]
    selected["reject"] = (selected[outcome] > -np.log(alpha)).where(selected[outcome].notna())
    return selected, diagnostics


selected_frames, selection_diagnostics = [], []
for generator in generators_sel:
    selected, diagnostic = select_generator_draws(generator_results(generator), generator, SELECTION_METHOD)
    if not selected.empty:
        selected_frames.append(selected)
    if not diagnostic.empty:
        selection_diagnostics.append(diagnostic)
selected_generated = pd.concat(selected_frames, ignore_index=True)
selected_power = (selected_generated.groupby(["Learner", "H0_coef"])["reject"]
                  .mean().reset_index()
                  .rename(columns={"H0_coef": "Treatment effect", "reject": "Power"}))
selection_diagnostics = (pd.concat(selection_diagnostics, ignore_index=True)
                         if selection_diagnostics else pd.DataFrame())
if not selection_diagnostics.empty:
    print("Type-I-error selection (in-sample; requires independent validation):")
    print(selection_diagnostics[selection_diagnostics.selected].to_string(index=False))

selected_fig, selected_ax = plt.subplots(figsize=(10, 6))
selected_plot = pd.concat([selected_power, df_plot[df_plot.Learner.isin(statistical_methods + ["Theoretical (original)", "Theoretical (downsized)"])]], ignore_index=True)
for learner, style in styles.items():
    curve = selected_plot[selected_plot.Learner == learner].sort_values("Treatment effect")
    if not curve.empty:
        selected_ax.plot(curve["Treatment effect"], curve.Power, label=learner, color=style["color"], linestyle=style["ls"], marker=style["marker"], linewidth=2)
selected_ax.set(title=f"Selection: {SELECTION_LABELS[SELECTION_METHOD]}\nGenerative models: {GENERATOR_TEST}", xlabel="Treatment effect", ylabel="Type I error / Power", xlim=(-0.02, 1.02), ylim=(0, 1.05))
selected_ax.axhline(alpha, color="gray", linestyle=":")
selected_ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3, fontsize=10)
plt.show()
print(selected_power[np.isclose(selected_power["Treatment effect"], 0)].to_string(index=False))


In [ ]:
# Calibrated rejection thresholds: c_k is the empirical 95th percentile of
# |Z_k| under H0, calibrated separately for all-draw and selected-draw curves.
from scipy.stats import norm

baseline_logp_columns = {
    "Log-rank": "log_pvalue_init",
    "Adjusted_Log-rank": "log_pvalue_adjusted_logrank_init",
    "Prog_Adjusted_Log-rank": "log_pvalue_prog_adjusted_logrank_init",
    "Cox_Wald_X": "log_pvalue_cox_wald_x_init",
    "Cox_Wald_Prog": "log_pvalue_cox_wald_prog_init",
}


def absolute_z_from_log_pvalues(log_pvalues):
    log_pvalues = pd.to_numeric(log_pvalues, errors="coerce")
    p_values = np.exp(-np.maximum(log_pvalues, 0))
    p_values = np.clip(p_values, np.finfo(float).tiny, 1.0)
    return norm.isf(p_values / 2)


selection_panels = [("All draws", df_res, None), (f"Selected: {SELECTION_METHOD}", selected_generated, SELECTION_METHOD),]
calibration_rows = []

for method, column in baseline_logp_columns.items():
    if method not in statistical_methods:
        continue
    if column not in baseline_results.columns:
        continue
    baseline_z = pd.Series(absolute_z_from_log_pvalues(baseline_results[column]), index=baseline_results.index,)
    null_z = baseline_z[np.isclose(baseline_results["H0_coef"], 0.0)].dropna()
    if null_z.empty:
        continue
    cutoff = float(null_z.quantile(0.95))
    for panel, _, _ in selection_panels:
        calibration_rows.extend(
            {"panel": panel, "Learner": method, "Treatment effect": effect, "abs_z": z_value, "cutoff": cutoff} for effect, z_value in zip(baseline_results["H0_coef"], baseline_z) if pd.notna(z_value)
        )

for panel, panel_results, selection_method in selection_panels:
    for generator in generators_sel:
        if generator not in styles:
            continue
        log_p_column = generator_test_column(generator)
        if selection_method is None:
            source_df = generator_results(generator)
            test_results = source_df.copy()
        else:
            test_results = panel_results[panel_results["Learner"] == generator].copy()
        if log_p_column not in test_results.columns or test_results.empty:
            continue

        test_results["abs_z"] = absolute_z_from_log_pvalues(test_results[log_p_column])
        null_z = test_results.loc[np.isclose(test_results["H0_coef"], 0.0), "abs_z"].dropna()
        if null_z.empty:
            continue
        cutoff = float(null_z.quantile(0.95))
        calibration_rows.extend(
            {"panel": panel, "Learner": generator, "Treatment effect": effect, "abs_z": z_value, "cutoff": cutoff}
            for effect, z_value in zip(test_results["H0_coef"], test_results["abs_z"]) if pd.notna(z_value)
        )

calibrated_results = pd.DataFrame(calibration_rows)
if calibrated_results.empty:
    raise ValueError("No null test statistics were available for calibration.")

calibrated_results["reject"] = calibrated_results["abs_z"] > calibrated_results["cutoff"]
calibrated_power = (
    calibrated_results.groupby(["panel", "Learner", "Treatment effect"])["reject"]
    .mean()
    .reset_index(name="Power")
)
calibration_cutoffs = (
    calibrated_results.groupby(["panel", "Learner"])["cutoff"]
    .first()
    .reset_index()
)

calibrated_fig, calibrated_axes = plt.subplots(1, len(selection_panels), figsize=(7 * len(selection_panels), 5.8), sharey=True)
for ax_cal, (panel, _, _) in zip(calibrated_axes, selection_panels):
    panel_power = calibrated_power[calibrated_power["panel"] == panel]
    logrank_cutoff = calibration_cutoffs.loc[(calibration_cutoffs["panel"] == panel) & (calibration_cutoffs["Learner"] == "Log-rank"), "cutoff"].iloc[0]
    theoretical_alpha = 2 * norm.sf(logrank_cutoff)
    theoretical_data = []
    for effect, row in df_theo.iterrows():
        theoretical_cases = {
            "Theoretical (original)": row["D_control"],
            "Theoretical (downsized)": row["D_control"] * aug_perc,
        }
        for learner, n_control in theoretical_cases.items():
            theoretical_data.append({
                "Learner": learner,
                "Treatment effect": effect,
                "Power": cpower(n_control, row["D_treated"], row["H0_coef_univ"], theoretical_alpha),
            })
    theoretical_data = pd.DataFrame(theoretical_data)

    for learner, style in styles.items():
        if learner.startswith("Theoretical"):
            data = theoretical_data[theoretical_data["Learner"] == learner].sort_values("Treatment effect")
        else:
            data = panel_power[panel_power["Learner"] == learner].sort_values("Treatment effect")
        if data.empty:
            continue
        ax_cal.plot(
            data["Treatment effect"], data["Power"], label=learner,
            color=style["color"], linestyle=style["ls"], marker=style["marker"],
            linewidth=2.0, markersize=6,
        )

    ax_cal.set_title(panel, fontsize=13, fontweight="semibold")
    ax_cal.set_xlabel(r"Treatment effect ($\beta$)", fontsize=13)
    ax_cal.set_xlim(-0.02, 1.02)
    ax_cal.set_ylim(0, 1.05)
    ax_cal.set_xticks(np.arange(0, 1.1, 0.2))
    ax_cal.set_yticks(np.arange(0, 1.1, 0.2))
    ax_cal.tick_params(axis="both", labelsize=11)
    ax_cal.grid(True, alpha=0.3)
    ax_cal.axhline(alpha, color="gray", linestyle=":", linewidth=1, alpha=0.7)

calibrated_axes[0].set_ylabel("Rejection rate using calibrated $c_k$", fontsize=13)
handles, labels = [], []
for ax_cal in calibrated_axes:
    panel_handles, panel_labels = ax_cal.get_legend_handles_labels()
    for handle, label in zip(panel_handles, panel_labels):
        if label not in labels:
            handles.append(handle)
            labels.append(label)
calibrated_fig.legend(
    handles, labels, loc="lower center", bbox_to_anchor=(0.5, -0.12),
    ncol=5, fontsize=9, frameon=False,
)
calibrated_fig.suptitle(
    "Power with null-calibrated thresholds "
    r"($c_k = Q_{0.95}(|Z_k| \mid b=0)$)",
    fontsize=15, fontweight="semibold",
)
calibrated_fig.tight_layout(rect=(0, 0.12, 1, 0.92))
plt.show()

print("Empirical null-calibrated critical values:")
print(calibration_cutoffs.sort_values(["panel", "Learner"]).to_string(index=False, formatters={"cutoff": "{:.3f}".format}))
print(f"Theoretical curves use the calibrated Log-rank level: {theoretical_alpha:.4f}")

In [ ]:
# Null-distribution diagnostics for generated-vs-treated tests at H0_coef = 0.
# Empirical p-value CDFs are compared with Uniform(0, 1); equivalent |Z| CDFs
# are compared with the half-normal distribution implied by a standard normal Z.
null_generators = [
    # "HI-VAE_weibull",
    # "HI-VAE_piecewise",
    "HI-VAE_weibull_prior",
    "HI-VAE_piecewise_prior",
    "Surv-GAN",
    "Surv-VAE",
    "TabPFN-naive-full",
    "TabPFN-naive-control",
    "TabFORGE-naive",
]
null_data = {}

for generator in null_generators:
    source_df = generator_results(generator)
    logp_col = generator_test_column(generator)
    if logp_col not in source_df.columns:
        continue
    null_rows = source_df[np.isclose(source_df["H0_coef"], 0.0)]
    logp = pd.to_numeric(null_rows[logp_col], errors="coerce")
    p_values = np.exp(-np.maximum(logp, 0)).clip(
        lower=np.finfo(float).tiny, upper=1.0
    ).dropna()
    if not p_values.empty:
        null_data[generator] = p_values.to_numpy()

baseline_null = baseline_results[
    np.isclose(baseline_results["H0_coef"], 0.0)
]
baseline_p = np.exp(
    -np.maximum(pd.to_numeric(baseline_null["log_pvalue_init"], errors="coerce"), 0)
).clip(lower=np.finfo(float).tiny, upper=1.0).dropna().to_numpy()

if not null_data:
    raise ValueError("No generator p-values found at treatment effect 0.")

# Overlay every generator so their departures from the theoretical null are directly comparable.
fig_p, ax_p = plt.subplots(figsize=(11, 7))
for generator in null_generators:
    if generator not in null_data:
        continue
    values = np.sort(null_data[generator])
    ecdf = np.arange(1, len(values) + 1) / len(values)
    style = styles[generator]
    ax_p.step(
        values, ecdf, where="post", label=generator,
        color=style["color"], linestyle=style["ls"], linewidth=1.8,
    )

if len(baseline_p):
    baseline_sorted = np.sort(baseline_p)
    baseline_ecdf = np.arange(1, len(baseline_sorted) + 1) / len(baseline_sorted)
    ax_p.step(
        baseline_sorted, baseline_ecdf, where="post",
        color="#222222", linestyle="--", linewidth=2,
        label="Real-control baseline",
    )
ax_p.plot([0, 1], [0, 1], color="#B2182B", linestyle=":", linewidth=2,
          label="Uniform(0, 1)")
ax_p.set_title("Null p-value distributions at treatment effect 0", fontsize=15)
ax_p.set_xlabel("Null p-value")
ax_p.set_ylabel("Empirical CDF")
ax_p.set_xlim(0, 1)
ax_p.set_ylim(0, 1)
ax_p.grid(True, alpha=0.25)
ax_p.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
fig_p.tight_layout()
plt.show()

fig_z, ax_z = plt.subplots(figsize=(11, 7))
for generator in null_generators:
    if generator not in null_data:
        continue
    abs_z = np.sort(absolute_z_from_log_pvalues(-np.log(null_data[generator])))
    ecdf = np.arange(1, len(abs_z) + 1) / len(abs_z)
    style = styles[generator]
    ax_z.step(
        abs_z, ecdf, where="post", label=generator,
        color=style["color"], linestyle=style["ls"], linewidth=1.8,
    )

if len(baseline_p):
    baseline_abs_z = np.sort(absolute_z_from_log_pvalues(-np.log(baseline_p)))
    baseline_ecdf = np.arange(1, len(baseline_abs_z) + 1) / len(baseline_abs_z)
    ax_z.step(
        baseline_abs_z, baseline_ecdf, where="post",
        color="#222222", linestyle="--", linewidth=2,
        label="Real-control baseline",
    )
z_grid = np.linspace(0, 8, 500)
ax_z.plot(
    z_grid, 2 * norm.cdf(z_grid) - 1,
    color="#B2182B", linestyle=":", linewidth=2,
    label="Half-normal |N(0, 1)|",
)
ax_z.set_title("Null |Z| distributions at treatment effect 0", fontsize=15)
ax_z.set_xlabel("Equivalent |Z| from two-sided p-value")
ax_z.set_ylabel("Empirical CDF")
ax_z.set_xlim(left=0)
ax_z.set_ylim(0, 1)
ax_z.grid(True, alpha=0.25)
ax_z.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
fig_z.tight_layout()
plt.show()

print("Null draws per generator:")
for generator, values in null_data.items():
    print(f"{generator}: {len(values):,}")
print(f"Real-control baseline: {len(baseline_p):,}")

In [ ]:
# Assess whether each generator's null |Z| distribution is explained by a
# half-normal scale inflation. Bootstrap MC seeds as clusters, not individual draws.
scale_fit_inputs = {}
for generator in null_generators:
    source_df = generator_results(generator)
    null_rows = source_df[np.isclose(source_df["H0_coef"], 0.0)].copy()
    logp_col = generator_test_column(generator)
    if logp_col not in null_rows.columns or "MC_seed" not in null_rows.columns:
        continue
    null_rows["abs_z"] = absolute_z_from_log_pvalues(null_rows[logp_col])
    null_rows = null_rows.dropna(subset=["abs_z", "MC_seed"])
    if not null_rows.empty:
        scale_fit_inputs[generator] = null_rows[["MC_seed", "abs_z"]]

baseline_scale_rows = baseline_null[["MC_seed", "log_pvalue_init"]].copy()
baseline_scale_rows["abs_z"] = absolute_z_from_log_pvalues(
    baseline_scale_rows["log_pvalue_init"]
)
baseline_scale_rows = baseline_scale_rows.dropna(subset=["abs_z", "MC_seed"])
scale_fit_inputs["Real-control baseline"] = baseline_scale_rows[["MC_seed", "abs_z"]]

rng = np.random.default_rng(20261019)
n_bootstrap = 1000
scale_fit_summaries = []
scale_fit_distributions = {}

for label, frame in scale_fit_inputs.items():
    values = frame["abs_z"].to_numpy()
    grouped = frame.groupby("MC_seed")["abs_z"].agg(
        cluster_sum_sq=lambda x: np.square(x).sum(),
        cluster_count="size",
    )
    cluster_sums = grouped["cluster_sum_sq"].to_numpy()
    cluster_counts = grouped["cluster_count"].to_numpy()
    n_clusters = len(grouped)

    sigma_hat = float(np.sqrt(np.sum(cluster_sums) / np.sum(cluster_counts)))
    sampled_clusters = rng.integers(0, n_clusters, size=(n_bootstrap, n_clusters))
    bootstrap_sums = cluster_sums[sampled_clusters].sum(axis=1)
    bootstrap_counts = cluster_counts[sampled_clusters].sum(axis=1)
    bootstrap_sigma = np.sqrt(bootstrap_sums / bootstrap_counts)
    ci_low, ci_high = np.quantile(bootstrap_sigma, [0.025, 0.975])

    sorted_values = np.sort(values)
    empirical_cdf = np.arange(1, len(sorted_values) + 1) / len(sorted_values)
    fitted_cdf = 2 * norm.cdf(sorted_values / sigma_hat) - 1
    ks_distance = float(np.max(np.abs(empirical_cdf - fitted_cdf)))
    scale_fit_distributions[label] = (sorted_values, empirical_cdf, sigma_hat)
    scale_fit_summaries.append({
        "Test": label,
        "MC clusters": n_clusters,
        "Null draws": len(values),
        "Fitted sigma": sigma_hat,
        "CI 2.5%": ci_low,
        "CI 97.5%": ci_high,
        "ECDF-fit distance": ks_distance,
    })

scale_fit_table = pd.DataFrame(scale_fit_summaries)
fig_scale, ax_scale = plt.subplots(figsize=(11, 7))
for label, (sorted_values, empirical_cdf, sigma_hat) in scale_fit_distributions.items():
    style = styles.get(label, {})
    line_color = style.get("color", "#222222")
    line_style = style.get("ls", "--")
    ax_scale.step(
        sorted_values, empirical_cdf, where="post", color=line_color,
        linestyle=line_style, linewidth=1.5, alpha=0.8,
        label=f"{label}: empirical",
    )
    z_grid = np.linspace(0, max(6.0, float(sorted_values[-1])), 400)
    ax_scale.plot(
        z_grid, 2 * norm.cdf(z_grid / sigma_hat) - 1,
        color=line_color, linestyle="-", linewidth=1.8,
        label=f"{label}: fitted sigma={sigma_hat:.2f}",
    )

ax_scale.set_title("Does scale inflation explain the null |Z| distributions?", fontsize=15)
ax_scale.set_xlabel("Equivalent |Z| from two-sided p-value")
ax_scale.set_ylabel("Empirical CDF")
ax_scale.set_xlim(left=0)
ax_scale.set_ylim(0, 1)
ax_scale.grid(True, alpha=0.25)
ax_scale.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=8)
fig_scale.tight_layout()
plt.show()

print("Half-normal scale fit; confidence intervals bootstrap MC seeds:")
print(
    scale_fit_table.to_string(
        index=False,
        formatters={
            "Fitted sigma": "{:.3f}".format,
            "CI 2.5%": "{:.3f}".format,
            "CI 97.5%": "{:.3f}".format,
            "ECDF-fit distance": "{:.3f}".format,
        },
    )
)

### Interpreting the preliminary TabFORGE run

TabFORGE uses a pretrained feature encoder, with decoder/diffusion initialized from scratch (`checkpoint=None`, as reported for this run). The CSV does not record training configuration or loss histories, so convergence cannot be established from it. The current batch defaults are 10,000 optimizer steps, split equally between decoder and diffusion—not 10,000 epochs. More steps are not necessarily better; compare training budgets using held-out generative diagnostics before fixing the configuration for independent MC evaluation.

Only five independent MC trials are currently available, with 50 synthetic draws per trial. These 250 draws are not 250 independent trials. Selected-draw rejection rates therefore change in increments of 0.20. Null calibration and `type1` draw selection reuse the same null trials and are exploratory, especially with five MCs. A calibrated null rate near 0.05 is not independent evidence of type-I-error control. Scratch initialization may affect performance, but does not by itself imply worse results than the paper; this naive censored-survival task also differs from generic tabular benchmarks.

The table below compares methods on the MC seeds available for TabFORGE, with each seed weighted equally. Matching seed numbers provides a descriptive common subset; it does not establish identical training data across experiment families.


In [ ]:
tabforge_frame = generator_results("TabFORGE-naive")
tabforge_seeds = set(tabforge_frame.MC_seed.unique())
common_mc_rows = []
for learner in generators_sel:
    frame = generator_results(learner)
    column = generator_test_column(learner)
    if column not in frame:
        continue
    frame = frame[frame.MC_seed.isin(tabforge_seeds)].copy()
    values = pd.to_numeric(frame[column], errors="coerce")
    frame["reject"] = (values > -np.log(alpha)).where(values.notna())
    per_mc = frame.groupby(["H0_coef", "MC_seed"])["reject"].mean()
    summary_mc = per_mc.groupby(level="H0_coef").agg(["mean", "min", "max", "count"])
    summary_mc["Learner"] = learner
    common_mc_rows.append(summary_mc.reset_index())
common_mc_summary = pd.concat(common_mc_rows, ignore_index=True).rename(columns={
    "mean": "Rejection rate", "min": "Min MC rate", "max": "Max MC rate", "count": "Valid MCs"})
print("Common-seed comparison (unselected draws; min/max are descriptive, not confidence intervals):")
print(common_mc_summary.to_string(index=False))
print("TabFORGE valid outcome counts and rejection rates per MC/effect:")
tf_values = pd.to_numeric(tabforge_frame[generator_test_column("TabFORGE-naive")], errors="coerce")
print(tabforge_frame.assign(reject=(tf_values > -np.log(alpha)).where(tf_values.notna()))
      .groupby(["H0_coef", "MC_seed"])["reject"].agg(["count", "mean"]).to_string())
